# Systematic Trend Following — ETF Strategy

**Author:** Isaac Nicas | **Last updated:** June 2026 | **Status:** Paper trading live on IBKR

---

## Overview

This notebook implements a long/flat systematic trend-following strategy across a 12-asset ETF universe covering US and international equities, government bonds, gold, and currencies. It was designed to deliver equity-like returns with meaningfully lower drawdowns than a passive index allocation.

### Performance summary (2008-2026 backtest)

| Metric | This strategy | S&P 500 (SPY) | Nasdaq-100 (QQQ) |
|---|---|---|---|
| Ann. return (net) | **16.45%** | 12.66% | 17.78% |
| Sharpe ratio | **0.87** | 0.64 | 0.79 |
| Max drawdown | **-26.96%** | -51.85% | -49.40% |
| Worst single year | **-22.09%** | -36.24% | -40.79% |

### How the notebook is organised

| Cell | What it does |
|------|-------------|
| 1 | Install dependencies |
| 2 | Imports and plot styling |
| 3 | Universe and strategy configuration |
| 4 | Download price and VIX data via yfinance |
| 5 | Compute TSMOM and CS-Mom signals |
| 6 | Build regime filter and leverage scalar |
| 7 | Size positions with vol-targeting, fast-exit, and dead-band filter |
| 8 | Run backtest with realistic transaction costs |
| 9 | Export daily returns to CSV |
| 10 | Compute full performance metrics |
| 11 | Alpha/beta decomposition vs SPY and QQQ |
| 12 | Full visualisation dashboard |
| 13 | Monthly returns heatmap |
| 14 | Signal attribution analysis |
| 15 | Export returns and positions to CSV |
| 16 | Optional pyfolio tear sheet |

### Design principles

- **Long/flat only** — no short positions. Goes to cash rather than shorting during bear regimes.
- **Two signals, ERC-weighted** — TSMOM (44%) provides the absolute trend gate; CS-Mom (56%) adds cross-sectional relative strength.
- **Single regime gate** — SPY 200-day MA cross combined with a short/long VIX comparison.
- **Monthly rebalancing with tiered dead-band** — trades only when position change exceeds an asset-class threshold.
- **Signal-based fast-exit** — cuts exposure when SPY drops >5% over 10 days; re-entry driven by signal recovery.

---

*Paper trading implementation in `/paper_trading/`. Live results will be added as data accumulates.*

In [ ]:
# Cell 1: Install dependencies
# Pin versions to ensure reproducibility. Run once per Colab session.

!pip install -q --upgrade \
    pandas==2.2.2 \
    numpy==2.0.0 \
    scipy==1.14.1 \
    yfinance \
    matplotlib \
    pyfolio-reloaded

import warnings
warnings.filterwarnings('ignore')
print('Dependencies installed.')


In [ ]:
# Cell 2: Imports
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import rankdata
from numpy.linalg import lstsq
from datetime import datetime

try:
    import pyfolio_reloaded as pf
    HAS_PYFOLIO = True
except ImportError:
    HAS_PYFOLIO = False

plt.style.use('seaborn-v0_8-darkgrid')
COLORS = {
    'strat': '#2196F3',
    'spy'  : '#FF5722',
    'qqq'  : '#4CAF50',
    'gross': '#90CAF9',
}
print('Imports complete.')


In [ ]:
# Cell 3: Configuration
# All strategy parameters are defined here. Change values here rather than
# inside signal or sizing functions.
#
# Key parameters:
#   target_vol:       annualised portfolio volatility target (15%)
#   regime_floor:     minimum position scalar in any regime (60%)
#   w_tsmom / w_cs:   ERC-derived signal blend weights (44% / 56%)
#   fast_exit_*:      emergency drawdown protection parameters
#   dead_band_*:      minimum weight change to trigger a trade per asset class

UNIVERSE = {
    # Equities (primary alpha source — 70%+ gross target)
    'SPY': 'equity', 'QQQ': 'equity', 'IWM': 'equity',
    'XLK': 'equity', 'SMH': 'equity',
    'EEM': 'equity', 'EWJ': 'equity',
    # Diversifiers — capped at 30% gross (crash protection only)
    'TLT': 'bond',   'IEF': 'bond',
    'GLD': 'commodity',
    'FXE': 'fx',     'FXY': 'fx',
}

CFG = {
    'start_date'       : '2007-01-01',
    'end_date'         : datetime.today().strftime('%Y-%m-%d'),
    'trend_horizons'   : [21, 63, 126, 252],
    'vol_lookback'     : 63,
    'target_vol'       : 0.15,
    'equity_gross_cap' : 0.70,
    'divers_gross_cap' : 0.30,
    'max_leverage'     : 2.0,
    'tc_bps'           : 6,
    'vix_short'        : 63,
    'vix_long'         : 252,
    'equity_ma'        : 200,
    'rebal_freq'       : 'M',
    'dead_band_macro'  : 0.01,
    'dead_band_equity' : 0.02,
    'dead_band_lev'    : 0.04,
    'lev_etfs'         : {'TQQQ', 'UPRO', 'SQQQ', 'SPXU'},

    # ── Regime / leverage (FIX: separate bull_score from regime_scalar) ───────
    'regime_floor'     : 0.60,   # scalar lower bound [0.60, 1.0]
    'bull_leverage'    : 1.5,    # applied when RAW bull_score > threshold
    'bear_leverage'    : 0.8,    # applied when RAW bull_score <= threshold
    'bull_lev_thresh'  : 0.55,   # threshold applied to UNCLIPPED bull_score

    # ── Signal weights (FIX: ERC-derived, not arbitrary 80/20) ───────────────
    # Equal-Risk-Contribution weighting based on standalone vols:
    #   TSMOM standalone vol ~22%, CS-Mom standalone vol ~17.5%
    #   ERC weight ∝ 1/vol → TSMOM: 1/22=0.0455, CS: 1/17.5=0.0571
    #   Normalised → TSMOM: 44%, CS-Mom: 56%
    #   Rounded to interpretable values that sum to 1.0:
    'w_tsmom'          : 0.44,
    'w_cs'             : 0.56,

    # ── Fast-exit trigger ─────────────────────────────────────────────────────
    'fast_exit_lookback'   : 10,
    'fast_exit_threshold'  : -0.05,
    'fast_exit_floor'      : 0.60,
    # Re-entry: signal-based, not calendar-based (FIX)
    'reentry_spy_days'     : 5,    # SPY 5-day return must turn positive
}

tickers        = list(UNIVERSE.keys())
equity_tickers = [t for t, c in UNIVERSE.items() if c == 'equity']
divers_tickers = [t for t, c in UNIVERSE.items() if c != 'equity']

print(f'Universe: {len(tickers)} assets | {len(equity_tickers)} equity | '
      f'{len(divers_tickers)} diversifiers')
print(f'Signal weights: TSMOM={CFG["w_tsmom"]:.0%} CS-Mom={CFG["w_cs"]:.0%} '
      f'(ERC-derived)')
print(f'Leverage: bull={CFG["bull_leverage"]}x (score>{CFG["bull_lev_thresh"]}) '
      f'bear={CFG["bear_leverage"]}x (threshold on unclipped score)')
print(f'Re-entry: signal-based (SPY {CFG["reentry_spy_days"]}D return > 0)')

In [ ]:
# Cell 4: Download data
#
# Downloads adjusted closing prices for all 12 universe assets plus VIX.
# SPY and QQQ are downloaded separately as benchmark return series.
# Forward-fills up to 5 days to handle non-US trading holidays (e.g. EWJ).
# Assets with more than 20% missing data are dropped automatically.

print('Downloading price data...')

raw = yf.download(
    tickers + ['^VIX', 'SPY', 'QQQ'],
    start=CFG['start_date'], end=CFG['end_date'],
    auto_adjust=True, progress=False
)['Close']

vix   = raw['^VIX'].dropna()
spy_b = raw['SPY'].pct_change().dropna()
qqq_b = raw['QQQ'].pct_change().dropna()

prices = raw[tickers].ffill(limit=5)
prices = prices.loc[:, prices.isna().mean() < 0.20]

active_tickers = prices.columns.tolist()
active_equity  = [t for t in active_tickers if UNIVERSE.get(t) == 'equity']
active_divers  = [t for t in active_tickers if UNIVERSE.get(t) != 'equity']

returns    = prices.pct_change()
common_idx = prices.dropna(how='all').index
prices     = prices.loc[common_idx]
returns    = returns.loc[common_idx]

print(f'Loaded: {len(active_tickers)} assets | {len(prices)} trading days')
print(f'  Equities:     {active_equity}')
print(f'  Diversifiers: {active_divers}')
print(f'  Date range:   {prices.index[0].date()} to {prices.index[-1].date()}')


In [ ]:
# Cell 5: Signal computation
#
# Two signals are blended using ERC (Equal-Risk-Contribution) weights:
#
# TSMOM (44%): measures each asset's own trend.
#   For each lookback horizon h, compute the h-day return divided by
#   realised volatility (EWM std, annualised). Average across four
#   horizons [21, 63, 126, 252] days and clip to [-1, 1].
#   The 5-day skip avoids contamination from short-term reversals.
#
# CS-Mom (56%): ranks assets against each other.
#   For each date, rank all assets by their 126-day return and
#   normalise to [-1, 1]. Top-ranked assets get the strongest positive
#   signal; bottom-ranked get the strongest negative signal.
#
# Both signals are clipped to zero (long-flat only) in Cell 7.

def tsmom_signals(prices, horizons, vol_lookback=63):
    daily_ret = prices.pct_change()
    ewm_vol   = daily_ret.ewm(span=vol_lookback).std() * np.sqrt(252)
    ewm_vol   = ewm_vol.replace(0, np.nan).ffill()
    signal_list = []
    for h in horizons:
        ret_h = prices.pct_change(h).shift(5)
        signal_list.append(ret_h / ewm_vol)
    ensemble = sum(signal_list) / len(signal_list)
    return ensemble.clip(-2, 2) / 2

def cs_momentum_signals(prices, lookback=126):
    ret_h = prices.pct_change(lookback).shift(5)
    def rank_row(row):
        valid = row.dropna()
        if len(valid) < 2:
            return pd.Series(np.nan, index=row.index)
        ranks = pd.Series(rankdata(valid), index=valid.index)
        return (2*(ranks-1)/(len(valid)-1)-1).reindex(row.index)
    return ret_h.apply(rank_row, axis=1)

print('Computing signals...')
sig_tsmom = tsmom_signals(prices, CFG['trend_horizons'], CFG['vol_lookback'])
sig_cs    = cs_momentum_signals(prices)

def align(sig):
    return sig.reindex(prices.index).fillna(0)[active_tickers]

sig_tsmom = align(sig_tsmom)
sig_cs    = align(sig_cs)

combined_signal = CFG['w_tsmom'] * sig_tsmom + CFG['w_cs'] * sig_cs

coverage = (combined_signal.abs() > 0.01).mean().mean()
print(f'Signal coverage: {coverage:.1%}  (expect >80%)')
print(f'Mean equity signal today: {sig_tsmom[active_equity].iloc[-1].mean():.3f}')
print(f'Mean divers signal today: {sig_tsmom[active_divers].iloc[-1].mean():.3f}')


In [ ]:
# Cell 6: Regime filter
#
# Produces two independent outputs from the same underlying signals:
#
# regime_scalar [0.60, 1.0] — multiplies every position size.
#   Approaches 1.0 in confirmed uptrends with calm vol; floors at 0.60.
#
# leverage_scalar {0.8x or 1.5x} — sets the gross leverage ceiling.
#   Uses the UNCLIPPED raw_bull_score so it can genuinely reach the bear
#   state. If the clipped regime_scalar were used, it would always be
#   >= 0.60, making bull_lev_thresh of 0.55 always satisfied and the
#   0.8x bear leverage would never fire.
#
# Inputs:
#   price_bull: 1 when SPY > 200-day MA, else 0
#   vol_calm:   1 when short-term VIX average < long-term VIX average
#   raw_bull_score: 0.6 * price_bull + 0.4 * vol_calm, smoothed 10 days

def build_regime(vix, equity_price, vix_short=63, vix_long=252, ma_window=200,
                 floor=0.60, bull_lev=1.5, bear_lev=0.8, bull_thresh=0.55):

    vix_al = vix.reindex(equity_price.index).ffill()
    ma     = equity_price.rolling(ma_window).mean()

    price_bull = (equity_price > ma).astype(float)

    vix_short_ma = vix_al.rolling(vix_short).mean()
    vix_long_ma  = vix_al.rolling(vix_long).mean()
    vol_calm     = (vix_short_ma < vix_long_ma).astype(float)

    # Raw bull score: unclipped [0, 1] — used for leverage switch
    raw_bull_score = (0.6 * price_bull + 0.4 * vol_calm).rolling(10).mean()

    # Regime scalar: clipped to [floor, 1.0] — used for position scaling
    regime_scalar = (floor + (1.0 - floor) * raw_bull_score).clip(floor, 1.0)

    # Leverage scalar: uses UNCLIPPED score so bear state genuinely fires
    lev_values     = raw_bull_score.apply(lambda x: bull_lev if x > bull_thresh else bear_lev)
    leverage_scalar = lev_values.rolling(5).mean()

    return regime_scalar, leverage_scalar, raw_bull_score


spy_price = prices['SPY'] if 'SPY' in prices.columns else prices[active_equity[0]]

regime_scalar, leverage_scalar, raw_bull_score = build_regime(
    vix, spy_price,
    CFG['vix_short'], CFG['vix_long'], CFG['equity_ma'],
    CFG['regime_floor'], CFG['bull_leverage'],
    CFG['bear_leverage'], CFG['bull_lev_thresh']
)

regime_scalar   = regime_scalar.reindex(prices.index).ffill().fillna(CFG['regime_floor'])
leverage_scalar = leverage_scalar.reindex(prices.index).ffill().fillna(1.0)
raw_bull_score  = raw_bull_score.reindex(prices.index).ffill().fillna(0.5)

bull_days = int((raw_bull_score > CFG['bull_lev_thresh']).sum())
bear_days = int((raw_bull_score <= CFG['bull_lev_thresh']).sum())
bear_pct  = bear_days / len(raw_bull_score)

print(f'Regime scalar  | mean={regime_scalar.mean():.3f}  min={regime_scalar.min():.3f}  max={regime_scalar.max():.3f}')
print(f'Raw bull score | mean={raw_bull_score.mean():.3f}  min={raw_bull_score.min():.3f}  max={raw_bull_score.max():.3f}')
print(f'Leverage state | bull={bull_days} days ({1-bear_pct:.1%})  bear={bear_days} days ({bear_pct:.1%})')
print(f'Mean leverage  | {leverage_scalar.mean():.3f}x')

if bear_pct < 0.02:
    print('WARNING: Bear state firing <2% of days — check bull_lev_thresh')
if bear_pct > 0.40:
    print('WARNING: Bear state firing >40% of days — strategy may be too defensive')

In [ ]:
# Cell 7: Position sizing
#
# Converts signals into daily target weights through five sequential steps:
#
# 1. Vol-targeting: weight = signal * (target_vol / asset_vol)
#    Each asset sized to contribute equal risk. Clip lower=0 (long/flat).
#
# 2. Regime scalar: multiply all weights by regime_scalar (0.60-1.0).
#    Reduces gross exposure in bear regimes without going fully to cash.
#
# 3. Diversifier cap: if diversifiers > 30% of gross, scale them down.
#
# 4. Leverage cap: if gross > leverage_scalar, scale all positions down.
#
# 5. Tiered dead-band: suppress trades where weight change < threshold.
#    Asset-class specific: 1% macro, 2% equity, 4% leveraged ETFs.
#
# Fast-exit: if SPY drops >5% over 10 days, cut all positions to 60%.
# Re-entry requires SPY 5-day return positive AND TSMOM signal positive.
# Signal-based, not calendar-locked, to avoid missing V-shaped recoveries.

def monthly_rebal_mask(index):
    month = pd.Series(index.month, index=index)
    return month != month.shift(-1, fill_value=month.iloc[-1] + 1)

def tiered_dead_band(new_pos, old_pos, ticker_list, lev_etfs,
                     db_macro, db_equity, db_lev, universe_map):
    result = old_pos.copy()
    for t in ticker_list:
        if t not in new_pos.index:
            continue
        if t in lev_etfs:
            threshold = db_lev
        elif universe_map.get(t, 'equity') == 'equity':
            threshold = db_equity
        else:
            threshold = db_macro
        if abs(new_pos[t] - old_pos.get(t, 0.0)) > threshold:
            result[t] = new_pos[t]
    return result

def build_reentry_signal(returns, combined_signal,
                         spy_col='SPY', spy_days=5):
    """
    FIX: Signal-based re-entry after fast-exit.
    Returns True on days when BOTH:
      - SPY rolling `spy_days` return > 0  (market stabilising)
      - Combined TSMOM signal on SPY > 0   (trend confirming)
    Checked daily — no calendar dependency.
    """
    if spy_col not in returns.columns:
        spy_col = returns.columns[0]

    spy_roll   = (1 + returns[spy_col]).rolling(spy_days).apply(
        np.prod, raw=True) - 1
    spy_trend  = combined_signal[spy_col] if spy_col in combined_signal.columns \
                 else pd.Series(1.0, index=returns.index)

    reentry_ok = (spy_roll > 0) & (spy_trend > 0)
    return reentry_ok.fillna(False)

def fast_exit_scalar(returns, spy_col='SPY', lookback=10, threshold=-0.05):
    if spy_col not in returns.columns:
        spy_col = returns.columns[0]
    spy_roll = (1 + returns[spy_col]).rolling(lookback).apply(
        np.prod, raw=True) - 1
    spy_roll = spy_roll.fillna(0).rolling(2).mean()
    return (spy_roll < threshold).fillna(False)

def size_positions(signal, returns, regime_scalar, leverage_scalar,
                   target_vol, max_lev, universe_map,
                   equity_tickers, divers_tickers, divers_cap=0.30,
                   vol_lb=63, rebal_freq='M',
                   lev_etfs=None,
                   db_macro=0.01, db_equity=0.02, db_lev=0.04,
                   fast_exit_lookback=10, fast_exit_threshold=-0.05,
                   fast_exit_floor=0.60, reentry_spy_days=5):

    if lev_etfs is None:
        lev_etfs = set()

    asset_vol = returns.ewm(span=vol_lb).std() * np.sqrt(252)
    asset_vol = asset_vol.replace(0, np.nan).ffill()

    if rebal_freq == 'M':
        mask = monthly_rebal_mask(returns.index)
    else:
        week = pd.Series(
            returns.index.isocalendar().week.values, index=returns.index)
        mask = week != week.shift(-1, fill_value=week.iloc[-1] + 1)

    # Precompute daily series
    exit_triggered = fast_exit_scalar(
        returns, lookback=fast_exit_lookback,
        threshold=fast_exit_threshold)

    # FIX: signal-based re-entry (checked daily)
    reentry_ok = build_reentry_signal(
        returns, signal, spy_days=reentry_spy_days)

    positions    = pd.DataFrame(0.0, index=returns.index, columns=returns.columns)
    prev_pos     = pd.Series(0.0, index=returns.columns)
    in_fast_exit = False
    fast_exit_count = 0
    reentry_count   = 0

    for i, date in enumerate(returns.index):
        triggered_today = bool(exit_triggered.iloc[i])
        reentry_today   = bool(reentry_ok.iloc[i])

        # ── Fast-exit: fires any day, immediately cuts positions ──────────
        if triggered_today and not in_fast_exit:
            scaled = prev_pos * fast_exit_floor
            positions.loc[date] = scaled
            prev_pos = scaled.copy()
            in_fast_exit = True
            fast_exit_count += 1
            continue

        # ── FIX: signal-based re-entry (daily check, not monthly calendar) ──
        if in_fast_exit:
            if reentry_today:
                # Signal confirms recovery — rebuild on next rebal day
                # or immediately if today IS a rebal day
                in_fast_exit = False
                reentry_count += 1
                if not mask.iloc[i]:
                    # Not a rebal day: hold cut positions one more day,
                    # rebuild at next rebal
                    positions.loc[date] = prev_pos
                    continue
                # Falls through to normal rebal logic below
            else:
                # Still in fast-exit, signal not confirmed — hold cut positions
                positions.loc[date] = prev_pos
                continue

        # ── Normal path: only act on rebalance days ───────────────────────
        if not mask.iloc[i]:
            positions.loc[date] = prev_pos
            continue

        av = asset_vol.loc[date].replace(0, np.nan)
        if av.isna().all():
            positions.loc[date] = prev_pos
            continue

        sig  = signal.loc[date].fillna(0)
        r_sc = float(regime_scalar.loc[date])
        l_sc = float(leverage_scalar.loc[date])

        # Vol-target sizing — long/flat only
        raw = (sig * (target_vol / av)).fillna(0).clip(lower=0)

        # Apply regime scalar
        raw = raw * r_sc

        # Equity gross floor
        eq_gross    = raw[equity_tickers].sum() if equity_tickers else 0
        div_gross   = raw[divers_tickers].sum()  if divers_tickers else 0
        total_gross = eq_gross + div_gross
        if total_gross > 0 and div_gross / (total_gross + 1e-10) > divers_cap:
            scale_div = (divers_cap * total_gross) / (div_gross + 1e-10)
            for t in divers_tickers:
                if t in raw.index:
                    raw[t] *= scale_div

        # Portfolio leverage cap using UNCLIPPED leverage scalar (FIX active)
        gross = raw.abs().sum()
        effective_cap = min(l_sc, max_lev)
        if gross > effective_cap:
            raw = raw * (effective_cap / gross)

        # Tiered dead-band
        raw = tiered_dead_band(
            raw, prev_pos, returns.columns.tolist(),
            lev_etfs, db_macro, db_equity, db_lev, universe_map)

        positions.loc[date] = raw
        prev_pos = raw.copy()

    return positions, fast_exit_count, reentry_count

print('Sizing positions (ERC weights + fixed regime + signal re-entry)...')
positions, n_exits, n_reentries = size_positions(
    combined_signal, returns, regime_scalar, leverage_scalar,
    CFG['target_vol'], CFG['max_leverage'], UNIVERSE,
    active_equity, active_divers, CFG['divers_gross_cap'],
    CFG['vol_lookback'], CFG['rebal_freq'],
    CFG['lev_etfs'],
    CFG['dead_band_macro'], CFG['dead_band_equity'], CFG['dead_band_lev'],
    fast_exit_lookback=CFG['fast_exit_lookback'],
    fast_exit_threshold=CFG['fast_exit_threshold'],
    fast_exit_floor=CFG['fast_exit_floor'],
    reentry_spy_days=CFG['reentry_spy_days']
)

coverage  = (positions.abs() > 0.01).mean().mean()
avg_lev   = positions.abs().sum(axis=1).mean()
eq_frac   = (positions[active_equity].abs().sum(axis=1)
             / (positions.abs().sum(axis=1) + 1e-10))

print(f'Positions sized  | Coverage: {coverage:.1%} | Avg gross lev: {avg_lev:.2f}x')
print(f'Equity fraction  | {eq_frac.mean():.1%} of gross exposure')
print(f'Fast-exit events | {n_exits} triggers | {n_reentries} signal re-entries')
print(f'Avg days in exit | '
      f'{(len(positions) - coverage * len(positions)) / max(n_exits, 1):.1f} '
      f'days per episode (lower = faster signal recovery)')

if n_exits > 0 and n_reentries == 0:
    print('WARNING: Fast-exit fired but signal re-entry never triggered — '
          'check reentry_spy_days or combined_signal coverage on SPY')
if coverage < 0.5:
    print('WARNING: Low coverage — check signal and data alignment')

In [ ]:
# Cell 8: Backtest
#
# Applies the position series to returns with a one-day lag:
#   pos_lagged[t] * return[t+1]
# The lag means the signal computed using day t's close is applied
# starting day t+1 — no lookahead bias.
#
# Transaction costs: 6bps per unit of turnover (conservative for liquid ETFs).
# The first 252 days are excluded from performance measurement because
# the earliest signal values have insufficient lookback history.
def backtest(positions, returns, tc_bps=6):
    pos_lagged = positions.shift(1).fillna(0)
    gross_ret  = (pos_lagged * returns).sum(axis=1)
    turnover   = pos_lagged.diff().abs().sum(axis=1)
    tc_daily   = (tc_bps / 10_000) * turnover
    return gross_ret, gross_ret - tc_daily, tc_daily

gross_ret, net_ret, tc_daily = backtest(positions, returns, CFG['tc_bps'])

start_live = returns.index[252]
gross_ret  = gross_ret.loc[start_live:]
net_ret    = net_ret.loc[start_live:]
tc_daily   = tc_daily.loc[start_live:]

spy_b_bt   = spy_b.reindex(net_ret.index).fillna(0)
qqq_b_bt   = qqq_b.reindex(net_ret.index).fillna(0)

pct_nz = (net_ret.abs() > 1e-6).mean()
print(f'Backtest: {start_live.date()} to {net_ret.index[-1].date()}')
print(f'Non-zero return days: {pct_nz:.1%}  (target: >85%)')
print(f'Avg daily TC: {tc_daily.mean()*10000:.2f} bps')
print(f'Gross/net gap (annualised): {(gross_ret.mean()-net_ret.mean())*252:.2%}')


In [ ]:
# Cell 9: Export daily returns to CSV
# Used to build the interactive charts in the portfolio writeup.

export_df = pd.DataFrame({
    'strategy_net'  : net_ret,
    'strategy_gross': gross_ret,
    'spy'           : spy_b_bt,
    'qqq'           : qqq_b_bt,
})
export_df.to_csv('trend_v4_daily_returns.csv')
print(f'Exported {len(export_df)} daily rows to trend_v4_daily_returns.csv')
print(export_df.head())


In [ ]:
# Cell 9b: Download CSV (Colab only)
# Triggers a browser download in Google Colab.
# In a local environment the file is already saved from Cell 9.

try:
    from google.colab import files
    files.download('trend_v4_daily_returns.csv')
except ImportError:
    print('Not in Colab. File saved locally.')


In [ ]:
# Cell 10: Performance metrics
#
# Standard statistics for the strategy and both benchmarks.
# Up/down capture is measured vs QQQ as the primary equity benchmark.
def compute_metrics(ret, label):
    r = ret.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    if len(r) < 30 or r.std() == 0:
        return {'Label': label, 'Note': 'Insufficient data'}
    cum     = (1 + r).cumprod()
    ann_ret = r.mean() * 252
    ann_vol = r.std()  * np.sqrt(252)
    sharpe  = ann_ret / ann_vol
    roll_mx = cum.cummax()
    max_dd  = ((cum - roll_mx) / roll_mx).min()
    calmar  = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
    dn_vol  = r[r < 0].std() * np.sqrt(252)
    sortino = ann_ret / dn_vol if dn_vol > 0 else np.nan
    annual  = r.resample('YE').apply(lambda x: (1+x).prod()-1)
    # Up/down capture vs QQQ
    qqq_al  = qqq_b_bt.reindex(r.index)
    up_days = qqq_al > 0
    dn_days = qqq_al < 0
    up_cap  = r[up_days].mean() / qqq_al[up_days].mean() if up_days.sum() > 0 else np.nan
    dn_cap  = r[dn_days].mean() / qqq_al[dn_days].mean() if dn_days.sum() > 0 else np.nan
    return {
        'Label'         : label,
        'Ann. Return'   : f'{ann_ret:.2%}',
        'Ann. Vol'      : f'{ann_vol:.2%}',
        'Sharpe'        : f'{sharpe:.2f}',
        'Sortino'       : f'{sortino:.2f}',
        'Max DD'        : f'{max_dd:.2%}',
        'Calmar'        : f'{calmar:.2f}',
        'Best Year'     : f'{annual.max():.2%}',
        'Worst Year'    : f'{annual.min():.2%}',
        'Total Return'  : f'{cum.iloc[-1]-1:.2%}',
        'Up Capture QQQ': f'{up_cap:.1%}' if not np.isnan(up_cap) else 'n/a',
        'Dn Capture QQQ': f'{dn_cap:.1%}' if not np.isnan(dn_cap) else 'n/a',
    }

rows = [
    compute_metrics(net_ret,   'Strategy (Net)'),
    compute_metrics(gross_ret, 'Strategy (Gross)'),
    compute_metrics(spy_b_bt,  'SPY Buy & Hold'),
    compute_metrics(qqq_b_bt,  'QQQ Buy & Hold'),
]
summary = pd.DataFrame(rows).set_index('Label')
print('\n' + '='*70)
print('  PERFORMANCE SUMMARY')
print('='*70)
print(summary.T.to_string())
print('='*70)


print(f'  Ann. Return  :  2.45% → {net_ret.mean()*252:.2%}')
print(f'  Sharpe       :  0.24  → {(net_ret.mean()/net_ret.std())*np.sqrt(252):.2f}')
print(f'  Max DD       : -23.1% → {((1+net_ret).cumprod()/((1+net_ret).cumprod()).cummax()-1).min():.2%}')


In [ ]:
# Cell 11: Alpha and beta decomposition
#
# alpha: annualised excess return unexplained by the benchmark
# beta:  sensitivity to benchmark daily moves
# IR:    information ratio of the residual (alpha consistency)
# corr:  linear correlation with the benchmark
def alpha_beta(strat, bench, label):
    idx = strat.dropna().index.intersection(bench.dropna().index)
    y = strat.loc[idx].values
    X = np.column_stack([np.ones(len(idx)), bench.loc[idx].values])
    coef, *_ = lstsq(X, y, rcond=None)
    a_d, beta = coef
    resid = y - X @ coef
    ir    = (resid.mean() / resid.std()) * np.sqrt(252) if resid.std() > 0 else np.nan
    corr  = np.corrcoef(bench.loc[idx].values, strat.loc[idx].values)[0, 1]
    print(f'  vs {label:6s}: Alpha={a_d*252:.2%}/yr  Beta={beta:.3f}  '
          f'IR={ir:.2f}  Corr={corr:.3f}')

print('Alpha / Beta Decomposition')
print('-'*60)
alpha_beta(net_ret, spy_b_bt, 'SPY')
alpha_beta(net_ret, qqq_b_bt, 'QQQ')

# QQQ excess return (more useful than vs zero)
qqq_excess = net_ret - qqq_b_bt
print(f'\nQQQ excess return (net - QQQ daily):')
print(f'  Ann. excess: {qqq_excess.mean()*252:.2%}')
print(f'  Tracking vol: {qqq_excess.std()*np.sqrt(252):.2%}')
ir_qqq = (qqq_excess.mean() / qqq_excess.std()) * np.sqrt(252)
print(f'  Info Ratio vs QQQ: {ir_qqq:.2f}')


In [ ]:
# Cell 12: Visualisation dashboard
#
# Six panels: cumulative returns (log scale), drawdowns, rolling Sharpe,
# annual bar chart, regime and leverage over time, daily return scatter vs QQQ.
fig = plt.figure(figsize=(18, 22))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35)

def dd_series(r):
    c = (1+r).cumprod(); return (c - c.cummax()) / c.cummax()

def roll_sh(r, w=252):
    return (r.rolling(w).mean() / r.rolling(w).std()) * np.sqrt(252)

# 1. Cumulative returns
ax1 = fig.add_subplot(gs[0, :])
((1+net_ret)  .cumprod()*100).plot(ax=ax1, color=COLORS['strat'], lw=2.5,
                                    label='Strategy (Net)')
((1+gross_ret).cumprod()*100).plot(ax=ax1, color=COLORS['gross'], lw=1.5,
                                    ls='--', alpha=0.7, label='Strategy (Gross)')
((1+spy_b_bt) .cumprod()*100).plot(ax=ax1, color=COLORS['spy'],   lw=2,
                                    label='SPY B&H')
((1+qqq_b_bt) .cumprod()*100).plot(ax=ax1, color=COLORS['qqq'],   lw=2,
                                    label='QQQ B&H')
ax1.set_yscale('log'); ax1.legend(fontsize=10)
ax1.set_title('Cumulative Returns — log scale (base=100)', fontsize=14, fontweight='bold')

# 2. Drawdowns
ax2 = fig.add_subplot(gs[1, :])
ax2.fill_between(net_ret.index, dd_series(net_ret), 0,
                  color=COLORS['strat'], alpha=0.45, label='Strategy')
dd_series(spy_b_bt).plot(ax=ax2, color=COLORS['spy'], lw=1.2, label='SPY')
dd_series(qqq_b_bt).plot(ax=ax2, color=COLORS['qqq'], lw=1.2, label='QQQ')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax2.set_title('Drawdowns', fontsize=13, fontweight='bold'); ax2.legend()

# 3. Rolling Sharpe
ax3 = fig.add_subplot(gs[2, 0])
roll_sh(net_ret) .plot(ax=ax3, color=COLORS['strat'], lw=2,   label='Strategy')
roll_sh(spy_b_bt).plot(ax=ax3, color=COLORS['spy'],   lw=1.5, label='SPY',   alpha=0.8)
roll_sh(qqq_b_bt).plot(ax=ax3, color=COLORS['qqq'],   lw=1.5, label='QQQ',   alpha=0.8)
ax3.axhline(0, color='black', lw=0.8, ls='--')
ax3.axhline(1, color='green', lw=0.8, ls=':')
ax3.set_title('Rolling 1Y Sharpe', fontsize=12, fontweight='bold'); ax3.legend(fontsize=9)

# 4. Annual returns
ax4 = fig.add_subplot(gs[2, 1])
ann_s   = net_ret .resample('YE').apply(lambda x: (1+x).prod()-1)
ann_spy = spy_b_bt.resample('YE').apply(lambda x: (1+x).prod()-1).reindex(ann_s.index)
ann_qqq = qqq_b_bt.resample('YE').apply(lambda x: (1+x).prod()-1).reindex(ann_s.index)
x = np.arange(len(ann_s)); w = 0.27
ax4.bar(x-w, ann_s.values,   w, color=COLORS['strat'], alpha=0.9, label='Strategy')
ax4.bar(x,   ann_spy.values, w, color=COLORS['spy'],   alpha=0.75, label='SPY')
ax4.bar(x+w, ann_qqq.values, w, color=COLORS['qqq'],   alpha=0.75, label='QQQ')
ax4.set_xticks(x); ax4.set_xticklabels(ann_s.index.year, rotation=45, fontsize=7)
ax4.axhline(0, color='black', lw=0.8)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax4.set_title('Annual Returns', fontsize=12, fontweight='bold'); ax4.legend(fontsize=8)

# 5. Regime + leverage
ax5 = fig.add_subplot(gs[3, 0])
ax5b = ax5.twinx()
regime_scalar.reindex(net_ret.index).plot(ax=ax5, color='purple', lw=1.2,
                                           label='Regime (L)')
leverage_scalar.reindex(net_ret.index).plot(ax=ax5b, color='steelblue', lw=1.2,
                                             ls='--', label='Leverage (R)')
ax5.set_ylim(0.4, 1.1); ax5b.set_ylim(0.5, 2.5)
ax5.set_title('Regime & Leverage (single gate)', fontsize=12, fontweight='bold')
ax5.legend(loc='upper left', fontsize=8); ax5b.legend(loc='upper right', fontsize=8)

# 6. Up/down capture
ax6 = fig.add_subplot(gs[3, 1])
qqq_up  = qqq_b_bt[qqq_b_bt > 0]
qqq_dn  = qqq_b_bt[qqq_b_bt < 0]
strat_on_up = net_ret.reindex(qqq_up.index)
strat_on_dn = net_ret.reindex(qqq_dn.index)
# Scatter: strategy return vs QQQ return
ax6.scatter(qqq_up.values*100, strat_on_up.values*100,
             alpha=0.15, s=4, color=COLORS['qqq'], label='QQQ up days')
ax6.scatter(qqq_dn.values*100, strat_on_dn.values*100,
             alpha=0.15, s=4, color=COLORS['spy'], label='QQQ down days')
xlim = max(abs(qqq_b_bt).quantile(0.99)*100, 3)
ax6.set_xlim(-xlim, xlim); ax6.set_ylim(-xlim, xlim)
ax6.axhline(0, color='black', lw=0.5); ax6.axvline(0, color='black', lw=0.5)
ax6.plot([-xlim,xlim],[-xlim,xlim], color='gray', lw=0.8, ls=':', label='1:1 line')
ax6.set_xlabel('QQQ daily return %'); ax6.set_ylabel('Strategy daily return %')
ax6.set_title('Strategy vs QQQ daily scatter', fontsize=12, fontweight='bold')
ax6.legend(fontsize=8)

fig.suptitle('Systematic Trend Following — Backtest Dashboard', fontsize=16,
              fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved')


In [ ]:
# Cell 13: Monthly returns heatmap
#
# Shows every calendar month's net return, colour-coded red to green.
# Useful for identifying seasonal patterns and understanding which specific
# periods drove the strategy's worst and best stretches.
def monthly_heatmap(ret, title):
    monthly = ret.resample('ME').apply(lambda x: (1+x).prod()-1)
    df = pd.DataFrame({'ret':monthly,'Year':monthly.index.year,
                        'Month':monthly.index.month})
    pivot = df.pivot(index='Year', columns='Month', values='ret')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                      'Jul','Aug','Sep','Oct','Nov','Dec']
    pivot['Annual'] = ret.resample('YE').apply(lambda x:(1+x).prod()-1).values
    fig, ax = plt.subplots(figsize=(16, max(5, len(pivot)*0.45)))
    vals = pivot.values.astype(float)
    im = ax.imshow(vals, cmap=plt.cm.RdYlGn, aspect='auto', vmin=-0.15, vmax=0.15)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    for r in range(vals.shape[0]):
        for c in range(vals.shape[1]):
            if not np.isnan(vals[r, c]):
                ax.text(c, r, f'{vals[r,c]:.1%}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.02)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    plt.tight_layout(); plt.show()

monthly_heatmap(net_ret, 'Strategy — Monthly Returns (Net)')


In [ ]:
# Cell 14: Signal attribution
#
# Evaluates each signal in isolation to understand its standalone contribution.
# Each signal is run through vol-targeting and long/flat clipping, but without
# the regime filter or dead-band, isolating signal quality from risk management.
#
# Note: the combined Sharpe may be lower than the best individual signal.
# TSMOM's crash protection role shows up in drawdown reduction, not Sharpe alone.
print('Signal attribution (standalone, long/flat, vol-targeted)...')

def standalone_perf(sig, label, target_vol=0.15):
    av = returns.ewm(span=63).std() * np.sqrt(252)
    av = av.replace(0, np.nan).ffill()
    pos = (sig * (target_vol / av)).clip(lower=0).clip(upper=2.0)
    gross = (pos.shift(1).fillna(0) * returns).sum(axis=1)
    r = gross.loc[start_live:]
    ann_r = r.mean() * 252
    ann_v = r.std()  * np.sqrt(252)
    sh    = ann_r / ann_v if ann_v > 0 else np.nan
    print(f'  {label:12s}: Return={ann_r:.2%}  Vol={ann_v:.2%}  Sharpe={sh:.3f}')
    return r

r_tsmom   = standalone_perf(sig_tsmom,      'TSMOM only')
r_cs      = standalone_perf(sig_cs,         'CS-Mom only')
r_combined= standalone_perf(combined_signal, 'Combined')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pd.DataFrame({'TSMOM':r_tsmom,'CS-Mom':r_cs,'Combined':r_combined}).apply(
    lambda x: (1+x).cumprod()*100).plot(ax=axes[0], lw=2)
axes[0].set_title('Signal Standalone vs Combined', fontweight='bold')
axes[0].set_ylabel('Value (base=100)'); axes[0].set_yscale('log')

srs = pd.Series({'TSMOM': (r_tsmom.mean()/r_tsmom.std())*np.sqrt(252),
                  'CS-Mom':(r_cs.mean()/r_cs.std())*np.sqrt(252),
                  'Combined':(r_combined.mean()/r_combined.std())*np.sqrt(252)})
srs.plot(kind='bar', ax=axes[1], color=['#2196F3','#4CAF50','#FF5722'], alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Standalone Sharpe Ratios', fontweight='bold')
axes[1].set_xticklabels(srs.index, rotation=0)
plt.tight_layout(); plt.show()


In [ ]:
# Cell 15: Export results
#
# trend_v4_returns.csv   — daily returns for strategy, SPY, QQQ
# trend_v4_positions.csv — daily target weights per asset
pd.DataFrame({
    'Strategy_Net'  : net_ret,
    'Strategy_Gross': gross_ret,
    'SPY'           : spy_b_bt,
    'QQQ'           : qqq_b_bt,
}).to_csv('trend_v4_returns.csv')

positions.to_csv('trend_v4_positions.csv')

print('Exported: trend_v4_returns.csv | trend_v4_positions.csv')


In [ ]:
# Cell 16: Pyfolio tear sheet (optional)
#
# Generates a comprehensive institutional-style tear sheet including rolling
# returns, factor exposures, and stress tests. Requires pyfolio-reloaded.
# All key metrics are available in Cell 10 if pyfolio is not installed.
if HAS_PYFOLIO:
    pf.create_full_tear_sheet(net_ret, benchmark_rets=qqq_b_bt)
else:
    print('Install pyfolio-reloaded for full tear sheet. Metrics shown in Cell 9.')


---

## Design rationale

### Why these two signals

| Signal | Weight | Role |
|--------|--------|------|
| TSMOM | 44% | Absolute trend gate. Goes flat when an asset's own trend turns negative. Provides crash protection. |
| CS-Mom | 56% | Relative strength. Ranks assets against each other. Drives the tilt toward strongest performers. |

Weights derived from Equal-Risk-Contribution: each signal sized so a dollar allocated to each contributes equal volatility to the combined signal. TSMOM has higher standalone vol (~22% ann.) than CS-Mom (~17.5%), so it receives the lower weight.

### Key design choices

| Choice | Reason |
|--------|--------|
| Long/flat only | Shorting ETFs introduces borrowing costs and positive-beta drag during bull markets. Going to cash in bear regimes preserves upside convexity vs QQQ. |
| Single regime gate | One clean gate (SPY 200MA + VIX slope) outperforms three compounding gates, which together reduced effective exposure to ~30% on normal non-crisis days. |
| Monthly rebalancing | Reduces turnover without sacrificing signal quality. Trend signals on 1-3 month horizons do not require weekly refreshes. |
| Tiered dead-band | Leveraged ETFs move 3x per day vs macro ETFs. Asset-class-specific thresholds reflect this asymmetry. |
| Signal-based fast-exit re-entry | A calendar lock misses V-shaped recoveries. Signal-based re-entry allows re-engagement as soon as price and trend confirm recovery. |

### What was removed from earlier versions

| Removed | Reason |
|---------|--------|
| HRP position sizing | Allocated most weight to low-vol assets (bonds, FX) where TSMOM is weakest. |
| Correlation gate (avg rho > 0.65) | Fired on normal cross-asset windows, not just crises. Halved positions ~40% of the time. |
| Three-level dynamic leverage | Combined with the other two gates to produce ~0.29x effective exposure on ordinary days. |
| Kalman filter signal | Standalone Sharpe 0.208. Complexity without payoff. |
| Carry signal | Requires clean rate data across asset classes. Standalone Sharpe 0.357 does not justify implementation complexity. |

---

*Live execution code in `/paper_trading/`. Portfolio writeup at [isaacnicas.github.io/quant-portfolio](https://isaacnicas.github.io/quant-portfolio).*